In [2]:
import torch
import numpy as np

from nnsight import TransformersModel
from utils import ProcessedIDs
import json
import nnsight



In [ ]:
class Config:
    def __init__(self, model: TransformersModel) -> None:
        self.model = model
        self.model.eval()

        self.n_layers = model.config.n_layer
        self.n_heads = model.config.n_head
        self.n_embed = model.config.n_embd
        self.he_tk = model.tokenizer(" he")["input_ids"][0]
        self.she_tk = model.tokenizer(" she")["input_ids"][0]

        self.batch_size = 768

        wte = [self.model.transformer.wte]
        mlps = [self.model.transformer.h[i].mlp
                for i in range(self.n_layers)]
        self.modules = wte + mlps
        
        self.set_device()

    def set_device(self):
        self.use_cuda = torch.cuda.is_available()
        self.n_gpus = torch.cuda.device_count() if self.use_cuda else 0
        self.device = torch.device("cuda" if self.use_cuda else "cpu")
        
        print(f"Device: {self.device}")

        # if self.use_cuda:
        #     self.model = self.model.to(self.device)

        #     # Use multiple GPUs if available
        #     if self.n_gpus > 1:
        #         print(f"Using {self.n_gpus} GPUs")
        #         for i in range(self.n_gpus):
        #             print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

        #         self.model = torch.nn.DataParallel(self.model)
        #     else:
        #         print(f"Using 1 GPU: {torch.cuda.get_device_name(0)}")
        # else:
        #     print("Using CPU")


class SourceActivations:
    def __init__(self) -> None:
        self.cache = {}

    def __call__(
        self,
        cfg: Config,
        source_prompt: str,
        modules: list,
        position: int = 1,
    ):
        cache_key = (source_prompt, position)

        if cache_key in self.cache:
            return self.cache[cache_key]
    
        with torch.no_grad(), cfg.model.trace(source_prompt):
            saved_acts = nnsight.save([
                module.output[0, position, :].save()
                for module in modules
            ])
            
            

        activations = [
            activation.detach().cpu().clone()
            for activation in saved_acts
        ]

        self.cache[cache_key] = activations
        return activations


def patch_neurons(cfg: Config,
                  base,
                  source,
                  source_activations: SourceActivations):

    src_acts = source_activations(cfg, source, cfg.modules)

    neuron_effects = []

    for module_idx in range(len(cfg.modules)):
        module = cfg.modules[module_idx]
        src_out = src_acts[module_idx].to(cfg.device)

        module_effects = []

        for start in range(0, cfg.n_embed, cfg.batch_size):
            end = min(start + cfg.batch_size, cfg.n_embed)
            batch_idxs = list(range(start, end))

            len_batch = len(batch_idxs)
            bases = [base] * len_batch

            with torch.no_grad(), cfg.model.trace(bases):
                base_out = module.output

                rows = torch.arange(len_batch, device=cfg.device)
                neurons = torch.arange(start, end, device=cfg.device)

                base_out[rows, 1, neurons] = src_out[start:end]

                logits = cfg.model.output.logits[:, -1, :]

                m = (logits[:, cfg.she_tk] - logits[:, cfg.he_tk]).save()

            module_effects.extend(m.tolist())

        neuron_effects.append(module_effects)

    return np.array(neuron_effects)

In [5]:
# cfg = Config(gpt2)

# base = "The nurse said that"
# source = "The man said that"

# source_activations = SourceActivations()

# results = patch_neurons(cfg, base, source, source_activations)

# print(results.shape)
# print(results)

In [6]:
def run_one_condition(cfg: Config,
                      examples: list[dict],
                      condition: dict,
                      source_activations:SourceActivations):

    processed_ids = ProcessedIDs(condition["name"], examples)

    print(f"{condition['name']}: {len(processed_ids)} already saved; "
          f"{len(examples) - len(processed_ids)} remaining")

    for i, example in enumerate(examples):
        if processed_ids.contains(example["id"]):
            continue

        base = example[condition["base_key"]]
        source = example[condition["source_key"]]

        effects = patch_neurons(cfg=cfg,
                                base=base,
                                source=source,
                                source_activations=source_activations)
        

        processed_ids.save(example["id"], effects)

        if (i + 1) % 5 == 0:
            print(
                f"{condition['name']}: saved examples "
                f"{i + 1}/{len(examples)}"
            )

In [ ]:
CONDITIONS = [
    {
        "name": "man_direct",
        "base_key": "man_context",
        "source_key": "profession_context",
    },
    {
        "name": "man_indirect",
        "base_key": "profession_context",
        "source_key": "man_context",
    },
    {
        "name": "woman_direct",
        "base_key": "woman_context",
        "source_key": "profession_context",
    },
    {
        "name": "woman_indirect",
        "base_key": "profession_context",
        "source_key": "woman_context",
    },
]


def run_experiment():
    gpt2 = TransformersModel("openai-community/gpt2", dispatch=True)

    with open("./data/examples.json", "r") as f:
        examples = json.load(f)

    source_activations = SourceActivations()
    config = Config(gpt2)

    for condition in CONDITIONS:
        run_one_condition(cfg=config,
                          examples=examples,
                          source_activations=source_activations,
                          condition=condition)
        

run_experiment()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

... (2 outputs left out)

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loaded 2873 processed IDs.
man_direct: 2873 already saved; 0 remaining
Loaded 692 processed IDs.
man_indirect: 692 already saved; 2181 remaining
man_indirect: saved examples 695/2873
man_indirect: saved examples 700/2873
man_indirect: saved examples 705/2873
man_indirect: saved examples 710/2873
man_indirect: saved examples 715/2873
man_indirect: saved examples 720/2873
man_indirect: saved examples 725/2873
man_indirect: saved examples 730/2873
man_indirect: saved examples 735/2873
man_indirect: saved examples 740/2873
man_indirect: saved examples 745/2873
man_indirect: saved examples 750/2873
man_indirect: saved examples 755/2873
man_indirect: saved examples 760/2873
man_indirect: saved examples 765/2873
man_indirect: saved examples 770/2873
man_indirect: saved examples 775/2873
man_indirect: saved examples 780/2873
man_indirect: saved examples 785/2873
man_indirect: saved examples 790/2873
man_indirect: saved examples 795/2873
man_indirect: saved examples 800/2873
man_indirect: saved

: 